In [1]:
import numpy as np
import pandas as pd

### Gradient Descent for Logistic Regression

Gradient Descent is the standard first-order optimization algorithm. It starts at w = 0 and repeatedly nudges the weights in the direction that reduces the binary cross-entropy loss.

#### Loss Function: Binary Cross-Entropy

J(w) = -(1/m) * Σ [ y log ŷ + (1 - y) log (1 - ŷ) ] 

where ŷ = sigmoid(Xw) is the sigmoid of the linear score. Unlike MSE, this loss is convex in w, so gradient descent is guaranteed to reach the global minimum.

#### The Gradient & Update Rule

A remarkable result: even though we squash the output through a non-linear sigmoid, the gradient has the same form as linear regression. 

The only difference: instead of y, we use ŷ. In words: the average of (prediction error × feature value) across all samples.

All weights are updated simultaneously after computing the full-batch gradient.

---

#### Key Hyperparameters

| Hyperparameter | Too small | Too large |
| :--- | :--- | :--- |
| **α (learning rate)** | Converges very slowly | Diverges / overshoots |
| **iterations** | Underfits | Wastes compute (but won't hurt if α is right) |

---

#### 3 Ways to Implement It

Below we implement gradient descent three times, progressively eliminating explicit Python loops using NumPy vectorization:

| Version | Loops removed | Core idea |
| :--- | :--- | :--- |
| **3-Loop** | None | Fully explicit: iterate over epochs → features → samples |
| **2-Loop** | Sample loop | Use np.sum to aggregate over all m samples at once |
| **1-Loop** | Sample + feature loops | One matrix multiply updates all w simultaneously |

---

> **Coding Exercise**
> 
> Write a Python class that performs logistic regression using gradient descent. The class should take NumPy arrays `X` (features with a column of ones for the intercept) and `y` (binary target) as input, along with the learning rate `alpha` and the number of iterations. Return the learned coefficients (weights) as a NumPy array.
> 
> **Requirements**
> - Minimize the Binary Cross-Entropy loss function.
> - Initialize all weights to zero.
> - Use batch gradient descent (use all samples in each iteration).
> - The input matrix `X` has shape (m, n) where m is the number of training examples and n is the number of features (including the bias column of ones).
> - The target vector `y` has shape (m,) with values in {0, 1}.

### 3 Loops

In [44]:
def sigmoid(z: np.ndarray):
    return 1.0/(1.0+np.exp(-z))

class LogisticRegressionGD3:
    def __init__(self,alpha=0.01,iterations=1000):
        self.alpha=alpha
        self.iterations=iterations
        self.theta=None

    def fit(self,X: np.ndarray ,y: np.ndarray):
        n,m=X.shape
        self.theta=np.zeros((m,1))
        y=y.reshape(-1,1)

        for _ in range(self.iterations):
            for i in range(m):
                gradient_sum=0
                for j in range(n):
                    y_hat=sigmoid(X[j] @ self.theta)
                    gradient_sum+=(y_hat-y[j]) * X[j,i]
                gradient=1/n * gradient_sum
                self.theta[i]=self.theta[i] - self.alpha*gradient


        return self


    def predict(self,X: np.ndarray):
        return sigmoid(X @ self.theta).flatten()


        



In [45]:
X = np.array([[1, 0], [1, 1], [1, 2], [1, 3], [1, 4], [1, 5]])
y=np.array([0, 0, 1, 0, 1, 1])

model =LogisticRegressionGD3()
model.fit(X,y)

In [46]:
print('Coefficients: ',model.theta.flatten())

Coefficients:  [-1.04587159  0.57574867]


### 2 Loops

In [47]:
class LogisticRegressionGD2:
    def __init__(self,alpha=0.01,iterations=1000):
        self.alpha=alpha
        self.iterations=iterations
        self.theta=None

    def fit(self,X: np.ndarray ,y: np.ndarray):
        n,m=X.shape
        self.theta=np.zeros((m,1))
        y=y.reshape(-1,1)

        for _ in range(self.iterations):
            for i in range(m):
                y_hat= sigmoid(X @ self.theta)
                gradient=(1/n) * np.sum((y_hat - y) * X[:,i].reshape(-1,1))
                self.theta[i]=self.theta[i]-self.alpha * gradient


        return self


    def predict(self,X: np.ndarray):
        return sigmoid(X @ self.theta).flatten()

In [48]:
X = np.array([[1, 0], [1, 1], [1, 2], [1, 3], [1, 4], [1, 5]])
y=np.array([0, 0, 1, 0, 1, 1])

model2 =LogisticRegressionGD2()
model2.fit(X,y)

In [49]:
print('Coefficients: ',model2.theta.flatten())

Coefficients:  [-1.04587159  0.57574867]


In [56]:
class LogisticRegressionGD1:
    def __init__(self,alpha=0.01,iterations=1000):
        self.alpha=alpha
        self.iterations=iterations
        self.theta=None

    def fit(self,X: np.ndarray ,y: np.ndarray):
        n,m=X.shape
        self.theta=np.zeros((m,1))
        y=y.reshape(-1,1)

        for _ in range(self.iterations):
            y_hat= sigmoid(X @ self.theta)

            gradient=  (1/n) * X.T @ (y_hat-y)

            self.theta = self.theta - self.alpha * gradient 


        return self


    def predict(self,X: np.ndarray):
        return sigmoid(X @ self.theta).flatten()

In [67]:
X = np.array([[1, 9], [1, 4], [1, 2], [1, 1], [1, 11], [1, 16]])
y=np.array([0, 0, 1, 0, 1, 1])

model3 =LogisticRegressionGD1()
model3.fit(X,y)

In [68]:
print('Coefficients: ',model3.theta.flatten())

Coefficients:  [-0.75739663  0.13839064]


In [69]:
model3.predict(X)

array([0.61966325, 0.44921719, 0.38210683, 0.35000756, 0.68241673,
       0.81105098])

In [63]:
X1 = np.array([
    [1, 1, 1],
    [1, 1, 2],
    [1, 2, 1],
    [1, 2, 2],
    [1, 2, 3],
    [1, 3, 2],
    [1, 3, 3],
    [1, 4, 3],
    [1, 4, 4],
    [1, 3, 1]
])

y1 = np.array([0, 0, 0, 1, 0, 0, 1, 1, 1, 1])

In [70]:
model4 =LogisticRegressionGD1(alpha=0.1,iterations=2000)

model4.fit(X1,y1)

In [71]:
print('Coefficients: ',model4.theta.flatten())

Coefficients:  [-4.84413816  1.87507928  0.09487956]
